# GenOS 1차 의도분류 테스트

`guide.ipynb`의 `requests` 기반 GenOS 호출 방식을 따라 작성한 독립 실행 노트북입니다.

현재 범위:

1. 버전 프롬프트 파일 로드
2. Redis에서 조회했다고 가정한 최근 대화 이력 전달
3. 문맥과 오타를 고려한 질문 보정
4. 업무 무관/질문 불충분 예외 판정
5. 6개 서브에이전트 중 하나로 1차 의도분류

이 단계에서는 FastAPI, LangChain, LangGraph를 사용하지 않습니다.

In [ ]:
import getpass
import json
import os
from enum import Enum, StrEnum
from pathlib import Path

import requests
import yaml
from pydantic import BaseModel, ConfigDict, Field, model_validator

# guide.ipynb의 GenOS 직접 호출 방식을 따릅니다.
GENOS_URL = "https://genos.genon.ai"
SERVING_ID = 850
MODEL = "qwen/qwen3.7-flash"
# None이면 prompts/intent-classification/active.yaml의 버전을 사용합니다.
PROMPT_VERSION = None
REQUEST_TIMEOUT_SECONDS = 120

# 토큰을 노트북 파일에 저장하지 않습니다.
# 환경 변수에 값이 없으면 실행 시 화면에서 안전하게 입력받습니다.
BEARER_TOKEN = os.getenv("GENOS_BEARER_TOKEN")
BEARER_TOKEN = "ffabf121d76e4b59aefb524b83cce180"
# if not BEARER_TOKEN:
#     BEARER_TOKEN = getpass.getpass("GENOS bearer token: ")

BASE_URL = f"{GENOS_URL.rstrip('/')}/api/gateway/rep/serving/{SERVING_ID}"
MODELS_ENDPOINT = f"{BASE_URL}/v1/models"
CHAT_ENDPOINT = f"{BASE_URL}/v1/chat/completions"
HEADERS = {
    "Authorization": f"Bearer {BEARER_TOKEN}",
    "Content-Type": "application/json",
}

print("Models endpoint:", MODELS_ENDPOINT)
print("Chat endpoint:", CHAT_ENDPOINT)
print("Model:", MODEL)

## 1. GenOS 모델 조회

서빙 ID와 Bearer token이 정상인지 먼저 확인합니다.

In [ ]:
models_response = requests.get(
    MODELS_ENDPOINT,
    headers=HEADERS,
    timeout=30,
)
models_response.raise_for_status()
print(json.dumps(models_response.json(), indent=2, ensure_ascii=False))

## 2. 버전 프롬프트 로드

`active.yaml`과 선택 버전의 `manifest.yaml`을 읽고, manifest에 선언된 라우터 프롬프트와 모든 에이전트 프롬프트를 순서대로 결합합니다.

In [ ]:
prompt_root = Path.cwd() / "prompts" / "intent-classification"
active_config = yaml.safe_load(
    (prompt_root / "active.yaml").read_text(encoding="utf-8")
)
selected_prompt_version = PROMPT_VERSION or active_config["active_version"]
version_directory = prompt_root / selected_prompt_version
manifest = yaml.safe_load(
    (version_directory / "manifest.yaml").read_text(encoding="utf-8")
)

# manifest의 router 순서를 먼저 사용합니다.
prompt_paths = [
    version_directory / manifest["router"]["system_prompt"],
    version_directory / manifest["router"]["agents_prompt"],
]

# manifest의 agent_code 순서대로 모든 에이전트 프롬프트를 추가합니다.
prompt_paths.extend(
    version_directory / "agents" / f"{agent_code}.md"
    for agent_code in manifest["agent_code"]
)

# 향후 manifest에 명시되지 않은 Markdown 프롬프트가 추가되더라도 누락하지 않습니다.
declared_paths = {path.resolve() for path in prompt_paths}
additional_paths = sorted(
    path
    for path in version_directory.rglob("*.md")
    if path.resolve() not in declared_paths
)
prompt_paths.extend(additional_paths)

missing_paths = [str(path) for path in prompt_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"프롬프트 파일이 없습니다: {missing_paths}")

# 빈 파일도 목록에는 포함하되, 결합 문자열에서는 내용이 있는 프롬프트만 사용합니다.
prompt_parts = []
empty_prompt_paths = []
for path in prompt_paths:
    content = path.read_text(encoding="utf-8").strip()
    if content:
        relative_path = path.relative_to(version_directory).as_posix()
        prompt_parts.append(f"<!-- prompt: {relative_path} -->\n{content}")
    else:
        empty_prompt_paths.append(path.relative_to(version_directory).as_posix())

system_prompt = "\n\n---\n\n".join(prompt_parts)

print("Active version:", selected_prompt_version)
print("Manifest version:", manifest["version"])
print("Declared agent codes:", manifest["agent_code"])
print("All prompt files:")
for path in prompt_paths:
    print(" -", path.relative_to(version_directory).as_posix())
print("Combined non-empty prompt count:", len(prompt_parts))
print("Combined prompt characters:", len(system_prompt))
if empty_prompt_paths:
    print("WARNING - empty prompt files:")
    for relative_path in empty_prompt_paths:
        print(" -", relative_path)

## 3. Pydantic Structured Output JSON Schema

`manifest.yaml`의 에이전트 코드로 Enum을 동적으로 만들고, Pydantic 모델에서 생성한 JSON Schema를 GenOS 요청과 응답 검증에 동일하게 사용합니다.

In [ ]:
# 에이전트 코드는 manifest.yaml에서 동적으로 생성합니다.
AgentCode = Enum(
    "AgentCode",
    {code.upper(): code.upper() for code in manifest["agent_code"]},
    type=str,
)


class ClassificationType(StrEnum):
    AGENT = "AGENT"
    EMPTY_QUERY = "EMPTY_QUERY"
    OUT_OF_SCOPE = "OUT_OF_SCOPE"


class IntentClassificationOutput(BaseModel):
    # Structured Output strict 모드에서 정의되지 않은 필드를 거부합니다.
    model_config = ConfigDict(extra="forbid")

    refined_query: str = Field(
        min_length=1,
        description="오탈자와 대화 문맥을 반영해 보정한 최종 질문",
    )
    classification_type: ClassificationType = Field(
        description="정상 에이전트 분류 또는 예외 분류 유형",
    )
    agent_code: AgentCode | None = Field(
        description="AGENT 분류일 때 선택한 코드, 예외 분류이면 null",
    )


    @model_validator(mode="after")
    def validate_agent_code(self):
        if self.classification_type == ClassificationType.AGENT:
            if self.agent_code is None:
                raise ValueError("AGENT 분류에는 agent_code가 필요합니다.")
        elif self.agent_code is not None:
            raise ValueError("예외 분류의 agent_code는 null이어야 합니다.")
        return self


intent_json_schema = IntentClassificationOutput.model_json_schema()

# OpenAI 호환 API의 Structured Output 요청 형식입니다.
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "intent_classification",
        "strict": True,
        "schema": intent_json_schema,
    },
}

print(json.dumps(response_format, indent=2, ensure_ascii=False))

## 4. 대화 이력과 현재 질문 설정

현재는 Redis에서 조회했다고 가정한 list를 직접 입력합니다. 이후 앱에서는 이 부분을 Redis 조회 코드로 교체합니다.

In [ ]:
# Redis history 예시
# conversation_history = [
#     {"role": "user", "content": "RP 상품을 보고 있어요."},
#     {"role": "assistant", "content": "어떤 내용이 궁금하신가요?"},
# ]

conversation_history = []


# 분류할 현재 질문
user_question = "이번달 내 환산점수 어떻게되??"

# 프론트에서 사용자가 선택한 에이전트 코드입니다.
# 1차 의도분류 자체에는 사용하지 않고 결과 비교에만 사용합니다.
frontend_agent_code = "RP"

history_text = "\n".join(
    f"{item['role']}: {item['content']}"
    for item in conversation_history
)

user_prompt = f"""
이전 대화:
{history_text or '(이전 대화 없음)'}

현재 사용자 질문:
{user_question}
""".strip()

print(user_prompt)

## 5. GenOS 의도분류 호출

In [ ]:
request_body = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    "temperature": manifest.get("model", {}).get("temperature", 0),
    "response_format": response_format,
}

chat_response = requests.post(
    CHAT_ENDPOINT,
    headers=HEADERS,
    json=request_body,
    timeout=REQUEST_TIMEOUT_SECONDS,
)
chat_response.raise_for_status()
chat_payload = chat_response.json()

print(json.dumps(chat_payload, indent=2, ensure_ascii=False))

## 6. Pydantic 응답 검증

In [ ]:
raw_content = chat_payload["choices"][0]["message"]["content"].strip()

# 요청에 사용한 것과 동일한 Pydantic 모델로 응답을 다시 검증합니다.
# JSON 문법, 필수 필드, Enum 값, AGENT/예외별 agent_code 규칙이 적용됩니다.
validated_output = IntentClassificationOutput.model_validate_json(raw_content)
classification = validated_output.model_dump(mode="json")

agent_code = classification["agent_code"]
classification_type = classification["classification_type"]
refined_query = classification["refined_query"]

print(json.dumps(classification, indent=2, ensure_ascii=False))

## 7. 에이전트 코드 비교

현재 노트북 단계의 최종 반환값입니다. 이후 FastAPI 응답 모델과 LangGraph state로 옮길 수 있는 형태입니다.

In [ ]:
# 예외 분류는 에이전트 코드 비교 없이 고정답변으로 종료합니다.
if classification_type == "OUT_OF_SCOPE":
    fixed_answer = "죄송합니다. 해당 질문은 지원하는 업무 범위와 관련이 없습니다."
elif classification_type == "EMPTY_QUERY":
    fixed_answer = (
        "질문의 내용을 이해하기 어렵습니다. "
        "문의하려는 내용을 구체적으로 다시 작성해 주세요."
    )
else:
    fixed_answer = None

if classification_type != "AGENT":
    routing_result = {
        "status": "EXCEPTION",
        "pass": False,
        "intent": None,
        "classification_type": classification_type,
        "refined_query": refined_query,
        "frontend_agent_code": frontend_agent_code,
        "final_agent_code": None,
        "fixed_answer": fixed_answer,
        "confirmation": None,
    }
else:
    agent_code_matches = (
        agent_code.casefold() == frontend_agent_code.casefold()
    )

    if agent_code_matches:
        # 두 코드가 같으면 추가 확인 없이 즉시 PASS합니다.
        routing_result = {
            "status": "PASS",
            "pass": True,
            "intent": agent_code,
            "classification_type": classification_type,
            "refined_query": refined_query,
            "frontend_agent_code": frontend_agent_code,
            "final_agent_code": agent_code,
            "fixed_answer": None,
            "confirmation": None,
        }
    else:
        # 코드가 다르면 다음 처리를 진행하지 않고 프론트 OK를 기다립니다.
        routing_result = {
            "status": "NEED_HUMAN_CONFIRMATION",
            "pass": False,
            "intent": agent_code,
            "classification_type": classification_type,
            "refined_query": refined_query,
            "frontend_agent_code": frontend_agent_code,
            "final_agent_code": None,
            "fixed_answer": None,
            "confirmation": {
                "type": "AGENT_CODE_MISMATCH",
                "message": "선택한 에이전트와 질문 의도가 다릅니다. 변경하시겠습니까?",
                "frontend_agent_code": frontend_agent_code,
                "classified_agent_code": agent_code,
                "required_signal": "OK",
            },
        }

print(json.dumps(routing_result, indent=2, ensure_ascii=False))

## 8. Human-in-the-loop: 프론트 OK 신호

`routing_result.status`가 `NEED_HUMAN_CONFIRMATION`이면 실제 앱에서는 이 상태를 저장하고 응답을 종료합니다. 프론트 팝업에서 사용자가 OK를 누른 뒤 별도 요청으로 아래 신호를 전달한다고 가정합니다.

- `OK`: 1차 의도분류 코드를 승인하고 `PASS`
- `None`: 아직 응답이 없으므로 대기
- 그 외 값: 승인을 받지 못했으므로 `REJECTED`


In [ ]:
# 실제 FastAPI에서는 프론트의 두 번째 요청 body에서 받게 될 값입니다.
frontend_ok_signal = "OK"

if routing_result["status"] == "NEED_HUMAN_CONFIRMATION":
    if frontend_ok_signal == "OK":
        # 사용자가 마스터 에이전트의 분류 결과를 승인했습니다.
        # 최종 실행 코드는 1차 의도분류에서 나온 agent_code로 변경합니다.
        routing_result = {
            **routing_result,
            "status": "PASS",
            "pass": True,
            "final_agent_code": routing_result["intent"],
            "confirmation": {
                **routing_result["confirmation"],
                "received_signal": "OK",
                "approved": True,
            },
        }
    elif frontend_ok_signal is None:
        # 프론트 응답이 아직 없으므로 다음 단계로 진행하지 않습니다.
        routing_result = {
            **routing_result,
            "status": "WAITING_HUMAN_CONFIRMATION",
        }
    else:
        routing_result = {
            **routing_result,
            "status": "REJECTED",
            "pass": False,
            "confirmation": {
                **routing_result["confirmation"],
                "received_signal": frontend_ok_signal,
                "approved": False,
            },
        }

print(json.dumps(routing_result, indent=2, ensure_ascii=False))

## 권장 테스트 질문

`user_question`, `conversation_history`, `frontend_agent_code`를 변경하여 다시 실행합니다.

| 목적 | 이전 문맥 | 현재 질문 | 기대 결과 |
|---|---|---|---|
| RP 문맥·오타 보정 | RP 상품 문의 | `그럼 금니는 어때?` | `RP` |
| 실적 수수료 | 없음 | `이번 달 실적 수수료가 궁금해` | `PERFORMANCE_FEE` |
| 자격기준 | 없음 | `가입 자격 요건을 알려줘` | `QUALIFICATION` |
| 수수료기준 | 없음 | `수수료율 부과 기준은?` | `FEE_POLICY` |
| 상품안내 | 없음 | `상품 가입 방법을 알려줘` | `PRODUCT_GUIDE` |
| 태블릿 | 없음 | `태블릿에서 로그인이 안 돼` | `TABLET` |
| 업무 무관 | 없음 | `오늘 날씨 알려줘` | `OUT_OF_SCOPE` |
| 질문 불충분 | 없음 | `그거 어떻게 해` | `EMPTY_QUERY` |